# SPIKE — mask-free conditioning (decisive, ~10 min)

Overfits ONE PIPE pair at a fixed timestep to answer: does the mask-free
InputProj actually carry the SOURCE image into the SD3.5 transformer, or is the
32→16 channel squeeze too lossy?

**PASS** = loss collapses (collapse_ratio < 0.3): source conditioning is wired,
full train is worth it. **FAIL** = flat loss: the source signal isn't reaching
the prediction → must change how source enters (extra tokens, not channel concat).

Needs GPU + SD3.5 access. Cheap — run before committing to a 4-hour full train.

In [ ]:
import subprocess, sys, os
from pathlib import Path
REPO = Path('/kaggle/working/VIN')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/BDT-17/VIN.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'], check=True)
sys.path.insert(0, str(REPO))
print('repo at', subprocess.run(['git','-C',str(REPO),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())
subprocess.run([sys.executable,'-m','pip','install','-q','--force-reinstall','--no-deps',
                'transformers==4.46.3','tokenizers==0.20.3','huggingface_hub==0.25.2'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',
                'diffusers==0.31.0','accelerate==0.34.2','peft==0.13.2','datasets>=2.20',
                'safetensors>=0.4.3','sentencepiece','protobuf','pillow>=10','numpy'], check=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import transformers, torch
assert transformers.__version__ == '4.46.3'
from transformers.utils import FLAX_WEIGHTS_NAME
assert torch.cuda.is_available(); print('OK on', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
_local = Path('/kaggle/input/stable-diffusion-3-5-medium')
HF_TOKEN = None
if _local.exists():
    SD35_MODEL = str(_local)
else:
    SD35_MODEL = 'stabilityai/stable-diffusion-3.5-medium'
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        import os; HF_TOKEN = os.environ.get('HF_TOKEN')
    assert HF_TOKEN, 'Need HF_TOKEN or mounted SD3.5'
    from huggingface_hub import login; login(token=HF_TOKEN)

In [ ]:
from LoRA.train.spike_maskfree_edit import run_maskfree_spike
verdict = run_maskfree_spike(base_model_id=SD35_MODEL, hf_token=HF_TOKEN, steps=300)
verdict